# Blackwell PTQ Lab: BF16 → FP8 → NVFP4

**Audience:** intermediate/advanced developers building production inference systems on NVIDIA Blackwell.  
**Outcome:** produce an auditable, apples-to-apples comparison of a pinned Nemotron checkpoint in BF16, FP8 W8A8 and NVFP4 W4A4. We will measure quality, checkpoint/VRAM footprint, TTFT, TPOT, throughput, power, utilization and energy per generated token using the NVIDIA stack.

> This notebook is the instructor-facing control plane. Lightweight cells run in-process. Download, calibration, quantization and benchmarking run through project wrappers so failures are logged and GPU processes are completely released between precisions. Start in `DRY_RUN=True`; switch it off only on the prepared Blackwell host.

## 0. Experimental contract

The comparison changes **one primary variable: weight/activation precision**. Everything else is fixed: model revision, tokenizer, calibration IDs, prompts, output lengths, TensorRT-LLM runtime settings, TP=1, no speculative decoding, and BF16 KV cache. FP8 and NVFP4 use checked-in selective recipes. Each precision runs in a clean process.

The pinned source is `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16`: a hybrid Mamba/Transformer mixture-of-experts Nemotron with about 30B total and 3.5B active parameters. It fits a 96 GB RTX PRO 6000 Blackwell in BF16 while making compression effects tangible. NVIDIA Model Optimizer performs real PTQ and writes unified Hugging Face checkpoints; TensorRT-LLM AutoDeploy consumes those checkpoints directly.

### Profiles

| Profile | Purpose | Calibration | Evaluation | Repetitions |
|---|---|---:|---:|---:|
| `DEV_SMOKE` | fast control-flow check | 16 × 512 tokens | 30 MMLU-Pro + 20 GSM8K | 1 |
| `WORKSHOP_B200` | B200/B300 instructor run | 128 × 512 tokens | 300 MMLU-Pro + 100 GSM8K | 3 |
| `FULL` | extended evidence | 128 × 512 tokens | 1,000 MMLU-Pro + 250 GSM8K | 5 |

RTX PRO 6000 Blackwell is SM120, B200 is SM100, and B300 is SM103. The preflight accepts data-center Blackwell 10.x plus RTX Blackwell 12.0; DGX Spark SM121 is intentionally outside this x86_64 workshop runtime. NVFP4 is never emulated on unsupported GPUs. Before class, create the pinned project environment with `./scripts/bootstrap.sh` and launch this notebook with `./scripts/launch_notebook.sh`; do not install packages ad hoc from notebook cells.

In [ ]:
from __future__ import annotations

import json, os, shlex, subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ptq_workshop.artifacts import ArtifactLayout, initialize_run
from ptq_workshop.config import make_config
from ptq_workshop.numerics import (
    bf16_round, precision_storage_projection, quantization_error_metrics,
    quantize_fp8_e4m3fn, quantize_nvfp4,
)
from ptq_workshop.reporting import create_dashboard

PROFILE = os.getenv('PTQ_PROFILE', 'DEV_SMOKE')
DRY_RUN = os.getenv('PTQ_DRY_RUN', '1') != '0'
config = make_config(PROFILE, project_root=PROJECT_ROOT)

# Recipe files determine the immutable fingerprint. During a fresh checkout,
# the first dry-run can still inspect the profile before recipes are staged.
try:
    layout = ArtifactLayout.from_config(config)
    RUN_DIR = layout.run_dir
    if DRY_RUN:
        print('[dry-run] immutable run layout resolved; no directories were created')
    else:
        initialize_run(config, layout)
except FileNotFoundError as exc:
    RUN_DIR = config.artifact_root / f'pending-{PROFILE.lower()}'
    print(f'Fingerprint pending: {exc}')

print(json.dumps({
    'profile': PROFILE, 'dry_run': DRY_RUN, 'model': config.model_id,
    'revision': config.model_revision, 'modelopt': config.modelopt_version,
    'kv_cache_dtype': config.kv_cache_dtype, 'run_dir': str(RUN_DIR),
    'python': sys.executable,
}, indent=2))

### Safe wrapper runner

Commands are displayed before execution, never invoked through a shell, and gain `--dry-run` automatically. Set `PTQ_DRY_RUN=0` before launching Jupyter to authorize GPU stages.

In [ ]:
def run_wrapper(script: str, *arguments: str, allow_missing_in_dry_run: bool = True):
    path = PROJECT_ROOT / 'scripts' / script
    command = [sys.executable, str(path), *map(str, arguments)]
    if DRY_RUN and '--dry-run' not in command:
        command.append('--dry-run')
    print('$', shlex.join(command))
    if not path.is_file():
        if DRY_RUN and allow_missing_in_dry_run:
            print(f'[dry-run] wrapper will be available after project setup: {path}')
            return None
        raise FileNotFoundError(path)
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def variant_command(script: str, precision: str, *extra: str):
    return run_wrapper(
        script, '--profile', PROFILE, '--run-dir', str(RUN_DIR),
        '--precision', precision, *extra,
    )

## 1. What the formats actually store

Quantization follows the same visible sequence in every low-precision format: choose a scale, divide into the format's range, round to a representable value, saturate out-of-range values, and retain the scale needed to reconstruct an approximation. Calibration chooses those ranges; it does not update the trained weights.

- **BF16** uses 1 sign, 8 exponent and 7 explicit fraction bits. Normal finite values have the form `(-1)^s × (1 + m/128) × 2^(e-127)`; it preserves FP32's broad exponent range but rounds away the low 16 fraction bits, normally with round-to-nearest-even. It needs no external quantization scale and occupies 16 bits per stored value.
- **FP8 E4M3FN** uses 1 sign, 4 exponent and 3 fraction bits. Its finite grid includes subnormals down to `2^-9`, normals from `2^-6`, and a maximum magnitude of 448. After scaling, values round to the nearest grid point and values beyond ±448 saturate; a weight/activation tensor also needs its scale metadata. Thus an FP8 tensor is approximately, but not identically, 8 bits per value once scales and alignment are counted.
- **NVFP4** data uses the E2M1 grid `0, ±0.5, ±1, ±1.5, ±2, ±3, ±4, ±6`. It adds two-level scaling: an FP8 E4M3 scale for every 16 values and a tensor-level FP32 scale. Normalized values round to the nearest E2M1 point and saturate outside ±6. For a fully quantized tensor, data plus block scales cost `4 + 8/16 = 4.5` bits/value before the per-tensor scale, padding/alignment, metadata, and any BF16 modules are counted. Therefore **“4-bit” describes the payload format, not the exact checkpoint bytes per parameter**.

The simulation below is explanatory. Real results later come only from packed ModelOpt exports and native Blackwell kernels—not fake quantization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(config.seed)
weights = rng.normal(0, 0.7, 4096).astype(np.float32)
weights[[17, 991, 3200]] *= 12  # controlled outliers
simulated = {
    'BF16': bf16_round(weights),
    'FP8 E4M3': quantize_fp8_e4m3fn(weights).dequantized,
    'NVFP4': quantize_nvfp4(weights).dequantized,
}
errors = {name: quantization_error_metrics(weights, values) for name, values in simulated.items()}
errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, values in simulated.items():
    absolute_error = np.sort(np.abs(values - weights))
    axes[0].plot(np.linspace(0, 100, absolute_error.size), absolute_error, label=name)
axes[0].set(xlabel='Percentile', ylabel='Absolute error', title='Quantization error CDF')
axes[0].set_yscale('log'); axes[0].legend()
projection = precision_storage_projection(30_000_000_000, quantized_fraction=0.85)
axes[1].bar(list(projection), [value / 2**30 for value in projection.values()])
axes[1].set(ylabel='Projected weight storage (GiB)', title='30B model, 85% quantized')
fig.tight_layout(); plt.show()
print('NVFP4 includes one 8-bit block scale per 16 quantized weights; real exports also retain selected BF16 modules and metadata.')

## 2. Preflight: prove the host can run the experiment

The preflight records GPU name/UUID, compute capability, driver, CUDA, framework versions, memory, ECC, clocks, power limit and free disk. A live run must stop if CUDA is unavailable, the GPU is not SM100/SM120, disk is insufficient, or the device is busy. This protects the class from reporting NVFP4 emulation or contaminated measurements.

In [ ]:
run_wrapper(
    'run_profile.py', '--profile', PROFILE, '--run-dir', str(RUN_DIR),
    '--stage', 'preflight',
)

## 3. Freeze inputs before quantizing

A reproducible comparison downloads **only the pinned BF16 source**, freezes deterministic public calibration/evaluation rows, and hashes IDs/text. We never start FP8 from an FP8 checkpoint or NVFP4 from FP8: both candidates originate from the same BF16 revision.

Calibration estimates activation ranges; it does not update learned weights. Workshop calibration uses left-padded, bounded sequences and a fixed seed. Evaluation rows are disjoint. Hugging Face credentials are read from the environment and never written to artifacts.

In [ ]:
PREPARED_ROOT = PROJECT_ROOT / 'artifacts' / 'prepared'
prepare = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_assets.py'),
           '--profile', PROFILE, '--root', str(PREPARED_ROOT)]
print('$', shlex.join(prepare))
if DRY_RUN:
    print('[dry-run] skipped model and dataset downloads')
else:
    subprocess.run(prepare, cwd=PROJECT_ROOT, check=True)

## 4. BF16 baseline

BF16 is not quantized. The validation stage loads the pinned checkpoint with identical serving settings, performs deterministic no-thinking greedy smoke prompts, and captures cold-load time, post-load/peak VRAM and checkpoint bytes. These establish the denominator for compression, speedup and accuracy deltas.

In [ ]:
manifest = RUN_DIR / 'manifest.json'
if manifest.is_file():
    print(manifest.read_text())
else:
    print('[dry-run] the live run creates the BF16 source manifest; joint artifact validation runs after both PTQ exports')

## 5. FP8 PTQ with NVIDIA Model Optimizer

The checked-in `configs/recipes/fp8.yaml` is the executable specification: FP8 E4M3 weights/activations, max calibration, sensitive modules excluded, and BF16 KV cache. Its selectors use the live Transformers/ModelOpt namespace (`model.layers.*`), while the unified export restores checkpoint keys under `backbone.layers.*`. The validator therefore proves both sides: the exact controlled input/weight quantizers are disabled in `.quant_summary.txt`, and all 36 translated attention/preceding-Mamba tensors are physically BF16 in the packed checkpoint; `exclude_modules` alone is not accepted as proof. The wrapper drives the pinned ModelOpt 0.46.0rc0 `examples/hf_ptq/hf_ptq.py`, captures the resolved recipe/quantizer summary/log, and exports a unified Hugging Face checkpoint. ModelOpt 0.45 is intentionally not used: its release branch can calibrate NemotronH experts but omitted their unified-HF export, producing the exact `QuantNemotronHExperts` `NotImplementedError` observed during the live FP8 rehearsal.

In [ ]:
recipe = PROJECT_ROOT / 'configs' / 'recipes' / 'fp8.yaml'
print(recipe.read_text() if recipe.is_file() else '[dry-run] fp8.yaml not staged yet')
variant_command('quantize_variant.py', 'fp8')

## 6. NVFP4 PTQ on Blackwell

NVFP4 is a native Blackwell path, not a generic 4-bit integer substitute. The selective `configs/recipes/nvfp4.yaml` recipe uses E2M1 data with 16-value FP8-scaled blocks while retaining accuracy-sensitive paths in BF16. It starts again from the pinned BF16 source and uses the same calibration manifest and BF16 KV cache as FP8.

In [ ]:
recipe = PROJECT_ROOT / 'configs' / 'recipes' / 'nvfp4.yaml'
print(recipe.read_text() if recipe.is_file() else '[dry-run] nvfp4.yaml not staged yet')
variant_command('quantize_variant.py', 'nvfp4')

## 7. Validate native packed artifacts

Before claiming a speedup, validation must prove: (1) FP8/NVFP4 metadata names the expected algorithm, (2) both recipes have the intended exclusions and BF16 KV cache, (3) low-precision tensors/scales are present, (4) checkpoint bytes actually shrink, (5) tokenizer/source revision match BF16, and (6) each variant returns a deterministic smoke response. The analysis stage then loads representative BF16 source tensors and the corresponding serialized packed tensors, dequantizes through ModelOpt's public QTensor APIs, and persists error CDFs, scalar error metrics, and scale distributions. A fake-quantized BF16 checkpoint may illustrate numerical error but cannot support a performance claim.

In [ ]:
run_wrapper(
    'run_profile.py', '--profile', PROFILE, '--run-dir', str(RUN_DIR),
    '--stage', 'validate',
)
run_wrapper(
    'run_profile.py', '--profile', PROFILE, '--run-dir', str(RUN_DIR),
    '--stage', 'analyze',
)
for precision in ('fp8', 'nvfp4'):
    checkpoint = RUN_DIR / 'checkpoints' / precision
    metadata = checkpoint / 'hf_quant_config.json'
    print(f'\n{precision.upper()}: {metadata}')
    if metadata.is_file():
        print(json.dumps(json.loads(metadata.read_text()), indent=2)[:4000])
    analysis = RUN_DIR / 'manifests' / f'{precision}-tensor-analysis.json'
    if analysis.is_file():
        report = json.loads(analysis.read_text())
        print('Packed tensors analyzed:', [item['name'] for item in report['tensors']])

## 8. Accuracy: paired samples, deterministic decoding

MMLU-Pro measures broad knowledge/reasoning; GSM8K is a compact arithmetic reasoning probe. All variants see identical sample IDs and use reasoning disabled, greedy decoding, the same parser and paired bootstrap resamples. We report accuracy, parsed-answer rate, 95% confidence intervals and the percentage-point delta from BF16—not isolated cherry-picked generations.

### Published NVIDIA reference — context, not this lab's result

| Official checkpoint | Published MMLU-Pro | Treatment |
|---|---:|---|
| BF16 | 78.3 | source reference |
| FP8 | 78.1 | official quantized reference |
| NVFP4 | 77.4 | **QAD after initial PTQ** |

These values are shown to teach provenance, not to pre-fill the workshop dashboard. In particular, the official NVFP4 checkpoint underwent quantization-aware distillation after PTQ, whereas this lab measures pure PTQ. Different prompts, reasoning settings, sample sets or evaluation harnesses also prevent a direct like-for-like claim. Only the JSON/CSV artifacts produced below determine the workshop comparison.

In [ ]:
for precision in ('bf16', 'fp8', 'nvfp4'):
    variant_command('evaluate_variant.py', precision, '--task', 'all')

## 9. Performance: prefill and decode are different workloads

The benchmark uses fixed token IDs/lengths, ignores EOS, warms up before recording, and starts a fresh TensorRT-LLM AutoDeploy process per precision.

| Scenario | Input tokens | Output tokens | Concurrency | Measured requests |
|---|---:|---:|---:|---:|
| `interactive` | 512 | 128 | 1 | 16 |
| `rag_balanced` | 2,048 | 256 | 8 | 32 |
| `throughput` | 1,024 | 128 | 32 | 64 |
| `full_prefill` *(optional; `FULL` only)* | 8,192 | 64 | 8 | 32 |

`DEV_SMOKE` preserves the same input/output lengths and concurrency but uses 2, 8 and 32 measured requests respectively for the first three scenarios. The optional 8K prefill case is never silently enabled outside `FULL`.

Metrics include TTFT, TPOT, E2E latency, request throughput and output/total tokens per second. The runner repeats according to the profile, checks coefficient of variation, and flags unstable measurements rather than averaging them away.

In [ ]:
for precision in ('bf16', 'fp8', 'nvfp4'):
    variant_command('benchmark_variant.py', precision, '--scenario', 'all')

## 10. GPU telemetry: performance per watt and per GiB

The benchmark samples NVML every 100 ms from just before the measured interval through completion. Saved CSV contains power, SM/memory utilization, used VRAM, temperature and clocks where supported. Power is trapezoid-integrated to joules and divided by generated tokens; unsupported counters stay `N/A`. Cold-load measurements remain separate from steady state.

Interpretation matters: lower precision can reduce memory traffic and unlock concurrency even when a single tiny request does not saturate Tensor Cores. A speed result is invalid if the GPU throttled, another process was active, or variability stayed above the profile threshold.

In [ ]:
telemetry_files = sorted((RUN_DIR / 'telemetry').glob('*.csv')) if (RUN_DIR / 'telemetry').exists() else []
print('Telemetry artifacts:', *telemetry_files, sep='\n- ' if telemetry_files else '\n')
print('Sampling/energy integration is owned by benchmark_variant.py; the notebook never polls the GPU in dry-run mode.')

## 11. Artifact-only dashboard

Reporting deliberately cannot import CUDA, load a checkpoint or rerun a benchmark. It reads completed JSON/JSONL/CSV artifacts only, so the figures are reproducible and safe to rebuild on a laptop. Missing metrics omit the corresponding plot instead of inventing zeroes.

In [ ]:
if DRY_RUN:
    print('[dry-run] dashboard generation is skipped so the notebook creates no artifact files')
elif RUN_DIR.is_dir():
    dashboard = create_dashboard(RUN_DIR, RUN_DIR / 'figures')
    from IPython.display import Markdown, display
    display(Markdown(dashboard['markdown']))
    for name, figure in dashboard['figures'].items():
        print(name)
        display(figure)
else:
    print(f'No run artifacts yet: {RUN_DIR}. Execute the staged wrappers, then rerun this cell.')

### Decision rubric

- Choose **BF16** when its quality margin is material, memory is ample, or an operation lacks a validated low-precision kernel.
- Choose **FP8** when you want the conservative production default: roughly half-sized weights, broad kernel support and usually small quality movement.
- Choose **NVFP4** when Blackwell-native memory capacity/throughput is the binding constraint and the measured task-specific quality delta is acceptable. Selective BF16 islands are a feature, not a failure.

Never generalize one workload point. Prefill, decode and concurrency can choose different winners, and domain adaptation can change outlier/calibration behavior.

### Instructor interpretation prompts

1. Did checkpoint bytes and peak VRAM fall by the amount the format predicts? If not, which BF16 islands, scales, or runtime buffers explain the gap?
2. Is the gain in TTFT, TPOT, or concurrency? Relate it to compute saturation versus memory bandwidth instead of quoting one aggregate speedup.
3. Are quality deltas larger than paired confidence intervals, and is parsed-answer rate stable? A parser or prompt change is not a quantization regression.
4. Does lower average power also lower joules/token, or did a slower run merely draw less power?
5. Which precision satisfies the workload's explicit quality, latency, capacity and energy budgets?

### Troubleshooting before rerunning

- **Preflight rejects the GPU:** verify the visible device is SM100/SM120, the container has the host driver, and no other process owns material VRAM. Do not force NVFP4 emulation.
- **OOM during calibration/export:** confirm the selected profile, calibration batch size 1, free disk, and that the previous precision process exited. Reducing the profile is valid only if the new fingerprint is reported.
- **Export is not smaller:** inspect `hf_quant_config.json`, packed tensors and scale tensors; a fake-quantized BF16 export is not performance evidence.
- **Accuracy collapses:** first check prompt template, reasoning mode, parser and calibration/evaluation overlap; then inspect per-layer error and selective exclusions.
- **Numbers are noisy or unexpectedly slow:** discard cold starts, check clocks/throttling/concurrent processes, keep fixed token lengths, and require the profile's repetitions before drawing a conclusion.

## 12. Optional extensions

### FP8 KV cache
Repeat **all three** precisions with FP8 KV cache as a separate experiment fingerprint. It often unlocks longer context or larger batches, but mixing it into only one candidate confounds the primary precision comparison.

### Quality recovery: MSE, AutoQuantize, QAD/QAT
If pure PTQ misses the quality budget, first test a better calibration corpus, selective layers, NVFP4 MSE calibration or ModelOpt AutoQuantize. Quantization-Aware Distillation/Training can recover more accuracy but performs optimization and is outside this PTQ lab. NVIDIA's official Nemotron 3 Nano NVFP4 checkpoint underwent **QAD after its initial PTQ step**; its 77.4 MMLU-Pro reference is therefore not a pure-PTQ result. Never substitute published checkpoint scores for this experiment unless source revision, calibration/training method, prompt template, decoding and evaluation harness are identical. The only workshop deltas are paired measurements against the BF16 row in this run's artifacts.

### Profiling
On an instructor host with profiler permissions, add a short Nsight Systems capture and inspect whether native FP8/NVFP4 GEMM kernels dominate. Keep profiler runs separate from headline timing because tracing adds overhead.

## References

1. NVIDIA Model Optimizer, [Hugging Face PTQ example](https://github.com/NVIDIA/Model-Optimizer/tree/0.46.0rc0/examples/hf_ptq) (workshop pins official tag 0.46.0rc0 at commit `33d05b0c446f528914173041057050f6d135fbf4`; this is the first tag containing NVIDIA's complete NemotronH export fix).
2. NVIDIA Model Optimizer, [PTQ documentation](https://nvidia.github.io/Model-Optimizer/guides/1_quantization.html).
3. NVIDIA Model Optimizer, [unified Hugging Face checkpoint deployment](https://nvidia.github.io/Model-Optimizer/deployment/3_unified_hf.html).
4. TensorRT-LLM, [Quantization](https://nvidia.github.io/TensorRT-LLM/latest/features/quantization.html).
5. TensorRT-LLM, [`trtllm-bench`](https://nvidia.github.io/TensorRT-LLM/latest/commands/trtllm-bench.html) and [`trtllm-eval`](https://nvidia.github.io/TensorRT-LLM/latest/commands/trtllm-eval.html).
6. NVIDIA Transformer Engine, [NVFP4 format and two-level scaling](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/features/low_precision_training/nvfp4/nvfp4.html).
7. NVIDIA Technical Blog, [Introducing NVFP4 for Efficient and Accurate Low-Precision Inference](https://developer.nvidia.com/blog/introducing-nvfp4-for-efficient-and-accurate-low-precision-inference/).
8. NVIDIA, [CUDA GPU compute capability table](https://developer.nvidia.com/cuda/gpus) and [RTX PRO 6000 specifications](https://www.nvidia.com/en-us/products/workstations/professional-desktop-gpus/rtx-pro-6000/).
9. NVIDIA, [NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16).
10. NVIDIA, [official Nemotron 3 Nano NVFP4 checkpoint card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4) (external QAD-enhanced reference only; do not substitute its scores for this PTQ run).
11. NVIDIA Model Optimizer, [release changelog](https://nvidia.github.io/Model-Optimizer/reference/0_changelog.html) (distinguishes PTQ features from QAT/QAD workflows).
12. NVIDIA, [Nemotron 3 Nano technical resources](https://docs.nvidia.com/nemotron/latest/).

All web claims are contextual guidance. The final comparison must be based on this run's pinned manifests and saved raw artifacts.

## Preserve the executed notebook

After the live run, save this notebook in Jupyter, then archive that saved file without executing it again:

```bash
.venv/bin/python scripts/archive_notebook.py --run-dir <artifacts/runs/run-id>
```

The command writes a hash-addressed `.ipynb` plus an execution manifest under `<run-dir>/notebooks/`. It archives partial/error outputs before returning a failure, so a broken workshop session remains auditable; use `--allow-partial` only when that partial state is intentional.

## Acceptance checklist

- [ ] One pinned BF16 source revision feeds both PTQ candidates.
- [ ] Calibration/evaluation IDs and hashes are saved; no overlap.
- [ ] FP8/NVFP4 recipes preserve the controlled BF16 KV cache.
- [ ] Export metadata/storage proves real packed quantization.
- [ ] Representative packed tensors have BF16 reconstruction metrics, error CDFs, and scale distributions.
- [ ] Same samples, prompts, decoding and parser across precisions.
- [ ] Warmups are excluded; fixed input/output lengths are verified.
- [ ] At least the profile's measured repetitions are present and variability is reported.
- [ ] Telemetry spans every measured interval; throttled/contaminated runs are flagged.
- [ ] Dashboard can be rebuilt from JSON/JSONL/CSV alone.
- [ ] Parsed-answer acceptance is at least 98%; failures preserve raw predictions and stop the run.
- [ ] Accuracy regressions are highlighted, never silently averaged away.
- [ ] The saved executed notebook and its execution manifest are archived under the run directory.